In [ ]:
# Imports

# built-in

# local
import motiongen.data_handling as dh

# 3rd-party
from IPython.display import Markdown
import pandas as pd
import krippendorff
import plotly.graph_objects as go
import numpy as np

# CSL


In [ ]:
# Load Data
root = "../data/human_annotations"
dimensions = ["natural", "faithful", "variability"]

frames = dh.reorganize_per_dimension(root, dimensions=dimensions)

natural = dh.create_annotations_dataset(frames, "natural")
faithful = dh.create_annotations_dataset(frames, "faithful")
variability = dh.create_annotations_dataset(frames, "variability")

users = list(natural.columns)[1:]

# MoBERT
mobert_path = "../data/automatic_scores_pilot/mobert_results.csv"
mobert_UN_path = "../data/automatic_scores_pilot/mobert_UN_results.csv"
mobert_cor_path = "../data/automatic_scores_pilot/mobert_corrupted_results.csv"
mobert_og = dh.load_mobert(mobert_path)
mobert_un = dh.load_mobert(mobert_UN_path).iloc[:6]
mobert_corrupted = dh.load_mobert(mobert_cor_path)

mobert_og["sample"] = [
    f"sample{i:02d}_nat{j:02d}.mp4"
    for j in range(10)
    for i in range(6)
]
mobert_un["sample"] = [
    f"sampleUN_nat{j:02d}.mp4"
    for j in range(6)
]

mobert_corrupted["sample"] = [
    f"sample{i:02d}_cor{j:02d}.mp4"
    for j in range(10)
    for i in range(6)
]

mobert = pd.concat([mobert_og, mobert_un])


In [ ]:
# Load Data
root = "../data/human_annotations/3dmodels"
dimensions = ["natural"]

frames_3d = dh.reorganize_per_dimension(root, dimensions=dimensions)

natural_3d = dh.create_annotations_dataset(frames_3d, "natural")

## 3D renders vs Stick figures

In [ ]:
natural_3d.head()

annotators_3d = {
    "Annotator_3": "25946edb",  # Mat
    "Annotator_2": "abb8f9d7",  # Ant
    "Annotator_1": "cbf70837",  # Jor
}

annotators_stick = {
    "Annotator_3": "fb93944b",  # Mat
    "Annotator_2": "422f70a2",  # Ant
    "Annotator_1": "a536aa20",  # Jor
}

common_annotations = natural_3d.merge(natural, how="inner", on="sample")

table = f"|Dimension|Krippendorf's $\\alpha$|\n|-|-|"
for ann in ["Annotator_1", "Annotator_2", "Annotator_3"]:
    alpha = krippendorff.alpha(common_annotations.loc[:, [annotators_3d[ann], annotators_stick[ann]]].to_numpy().T, level_of_measurement="ordinal")
    table += f"\n|{ann}|{alpha:.2f}|"
display(Markdown(table))



In [ ]:
mat = common_annotations.loc[:, ["sample", annotators_3d["Annotator_3"], annotators_stick["Annotator_3"]]]
mat["diff"] = mat[annotators_3d[ann]] - mat[annotators_stick[ann]]
mat.describe()

In [ ]:
mat

## Annotator agreement over different dimensions

In [ ]:
# Annotator Agreement Computation

faith = krippendorff.alpha(faithful.drop(columns=["sample"]).to_numpy().T[1:], level_of_measurement="ordinal")
nat = krippendorff.alpha(natural.drop(columns=["sample"]).to_numpy().T[1:], level_of_measurement="ordinal")
var = krippendorff.alpha(variability.drop(columns=["sample"]).to_numpy().T[1:], level_of_measurement="ordinal")

display(Markdown(f"|Dimension|Krippendorf's $\\alpha$|\n|-|-|\n|Naturalness|{nat:.02f}|\n|Faithfulness|{faith:.02f}|\n|Variability|{var:.02f}|"))

All agreement values are bellow acceptance values (0.667); pointing torwards the difficulty of defining these carachteristics in a non-subjective way. Of the three, variability shows practically zero reliability in terms of human appreciation, posing this dimension as ill-defined. Here, the subjectivity issues are particularly relevant, as it seems that every annotator payed attention to completely independent factors when quantifying variability within the tested samples. Moreover, as plausible variability greatly depends on the specific type of motion being performed, this evaluation is highly affected by the prior beliefs of how variable the motion samples could be, against what they observe, and is susceptible to different interpretations of the associated prompt.

## Visualization of Human Annotations

In [ ]:
# Visualization code definitions

def jitter(nseg):
    """Create gaussian jitter to facilitate visualization of discrete data clusters.
    """
    return np.random.normal(0, .05, nseg)

def annotation_scatter(dimension):
    """Create plotly Figure that shows distributions of annotation values among annotators.

    Parameters
    ----------
    dimension : pd.Dataframe
        DataFrame with annotation values for each annotator for a given analysis dimension.
    
    Returns
    -------
    plotly.Graph_Objects.Figure
    """
    nseg = len(dimension)
    return go.Figure(data=[
        go.Scatter(
            x = dimension.loc[:, users[0]] + jitter(nseg)-0.1,
            y = dimension.loc[:, users[1]] + jitter(nseg),
            mode = "markers",
            name = F"{users[0]} vs {users[1]}"
        ),
        go.Scatter(
            x = dimension.loc[:, users[0]] + jitter(nseg)+0.1,
            y = dimension.loc[:, users[2]] + jitter(nseg),
            mode = "markers",
            name = F"{users[0]} vs {users[2]}"
        )
    ])

def parcat_annotations(dimension):
    """Create Parcat visualization of human annotations for a given dimension.

    Parameters
    ----------
    dimension : pd.Dataframe
        DataFrame with annotation values for each annotator for a given analysis dimension.
    
    Returns
    -------
    plotly.Graph_Objects.Figure
    """
    ann1_dim = go.parcats.Dimension(
        values=dimension[users[0]],
        categoryorder='category ascending', label=users[0]
    )
    ann2_dim = go.parcats.Dimension(
        values=dimension[users[1]],
        categoryorder='category ascending', label=users[1]
    )
    ann3_dim = go.parcats.Dimension(
        values=dimension[users[2]],
        categoryorder='category ascending', label=users[2]
    )


    return go.Figure(data = [go.Parcats(
        dimensions=[ann1_dim, ann2_dim, ann3_dim],
        hoveron='color', hoverinfo='count+probability',
        labelfont={'size': 18, 'family': 'Times'},
        tickfont={'size': 16, 'family': 'Times'},
        arrangement='freeform'
    )])

def filter_natural_segments(data, group):
    def gt_filter(x):
        return "GT" in x
    def un_filter(x):
        return "UN" in x
    def generated_filter(x):
        return (not gt_filter(x)) and (not un_filter(x))
    
    groups = {
        "GT": gt_filter,
        "UN": un_filter,
        "generated": generated_filter
    }
    filtered = data.loc[data["sample"].apply(groups[group])]
    return pd.melt(filtered, id_vars=["sample"])

def naturalness_violins():
    annotators_stick = {
        "fb93944b": "Ann. 3",  # Mat
        "422f70a2": "Ann. 2",  # Ant
        "a536aa20": "Ann. 1",  # Jor
    }
    natural_renamed = natural.rename(columns=annotators_stick)
    GT = filter_natural_segments(natural_renamed, "GT")
    UN = filter_natural_segments(natural_renamed, "UN")
    generated = filter_natural_segments(natural_renamed, "generated")

    return go.Figure(
        data=[
            go.Box(x=generated.variable, y=generated.value, name="Generated samples", boxpoints="all", pointpos=0, fillcolor="rgba(0,0,0,0)", line={"color": "rgba(0,0,0,0)"}, marker={"color": "#636EFA", "size":10}),
            go.Box(x=UN.variable, y=UN.value, name="Fandango samples", boxpoints="all", pointpos=-1, fillcolor="rgba(0,0,0,0)", line={"color": "rgba(0,0,0,0)"}, marker={"color": "#EF553B", "size":10}),
            go.Box(x=GT.variable, y=GT.value, name="Motion Capture samples", boxpoints="all", pointpos=1, fillcolor="rgba(0,0,0,0)", line={"color": "rgba(0,0,0,0)"}, marker={"color": "#00CC96", "size":10}),
        ],
        layout={
            "scattermode": "group",
            "yaxis": {"range": [0, 6], "tickvals": [1, 2, 3, 4, 5], "title": "Naturalness Score", "gridcolor": "gray"},
            "xaxis": {"title": "Annotator", "type": "category", "categoryorder":'category ascending', "gridcolor": "#DDDDDD"},
            "width": 500, "height": 500,
            "margin": {"l": 5},
            "plot_bgcolor": "white",
            "legend": {
                "yanchor": "top",
                "y": 1.25,
                "xanchor": "left",
                "x": 0.01
            },
            "margin": {"l": 5, "t": 10, "r": 20, "b": 0},
            "font": {"size": 17}
        }
    )


In [ ]:
with open("annotators_violins.svg", "wb") as fid:
    fid.write(naturalness_violins().to_image("svg"))

naturalness_violins()

### Naturalness

In [ ]:
display(annotation_scatter(natural))
display(parcat_annotations(natural))
display(naturalness_violins())

hicfalop = natural.loc[:, users].transpose().max()
human_natural = pd.DataFrame({"sample": natural["sample"], "naturalness_h": hicfalop})

### Faithfulness

In [ ]:
display(annotation_scatter(faithful))
display(parcat_annotations(faithful))

hicfalop = faithful.loc[:, users].transpose().max()
human_faithful = pd.DataFrame({"sample": faithful["sample"].apply(lambda x: x.replace("rep", "nat")), "faithfulness_h": hicfalop})

### Variability

In [ ]:
display(annotation_scatter(variability))
display(parcat_annotations(variability))

## Comparison with automatic measures

In [ ]:
human = pd.merge(human_natural, human_faithful, how="outer", on="sample")
all_data = pd.merge(human, mobert, how="outer", on="sample")

In [ ]:
fig = go.Figure(
    data=[
        go.Scatter(
            x=all_data.loc[~all_data["sample"].str.contains("UN"), "naturalness_h"],
            y=all_data.loc[~all_data["sample"].str.contains("UN"), "naturalness"],
            mode="markers", name="Generated samples", marker_size=10),
        go.Scatter(
            x=all_data.loc[all_data["sample"].str.contains("UN"), "naturalness_h"],
            y=all_data.loc[all_data["sample"].str.contains("UN"), "naturalness"],
            mode="markers", name="Fandango samples", marker_color="#EF553B", marker_size=10),
        go.Scatter(x=[0]*len(mobert_corrupted), y=mobert_corrupted["naturalness"], mode="markers", name="Corrupted samples", marker_color="#AB63FA", marker_size=10)
    ],
    layout={
        "yaxis": {"title": "MoBERT Naturalness Score", "gridcolor": "gray"},
        "xaxis": {"title": "Human Naturalness<br>Annotation", "range":[-0.2, 5.2], "dtick": 1, "gridcolor": "#DDDDDD"},
        "showlegend": True,
        "width": 250, "height": 500,
        "plot_bgcolor": 'white',
        "legend": {
            "yanchor": "top",
            "y": 1.30,
            "xanchor": "left",
            "x": 0.01
        },
        "margin": {"l": 5, "t": 10, "r": 20, "b": 0},
        "font": {"size": 17}
    }
)


with open("naturalness_scores_per_annotation_level.svg", "wb") as fid:
    fid.write(fig.to_image("svg"))

fig


There appears to be some level of alignment between 

In [ ]:
fig = go.Figure(
    data=[
        go.Scatter(x=all_data.faithfulness_h, y=all_data.faithfulness, mode="markers", name="Generated samples", marker_size=10),
        go.Scatter(x=[0]*len(mobert_corrupted), y=mobert_corrupted["faithfulness"], mode="markers", name="Corrupted samples", marker_color="#AB63FA", marker_size=10),
        go.Scatter(x=[None], y=[None], mode="markers", name="Fandango samples", marker_color="#EF553B", marker_size=10)
    ],
        layout={
        "yaxis": {"title": "MoBERT Faithfulness Score", "range": [0.28, 0.99], "gridcolor": "gray"},
        "xaxis": {"title": "Human Faithfulness<br>Annotation", "range": [-0.2, 5.2], "dtick": 1, "gridcolor": "#DDDDDD"},
        "showlegend": False,
        "width": 250, "height": 500,
        "plot_bgcolor": 'white',
        "legend": {
            "yanchor": "top",
            "y": 1.30,
            "xanchor": "left",
            "x": 0.01
        },
        "margin": {"l": 5, "t": 90, "r": 20, "b": 0},
        "font": {"size": 17}
    }
)


with open("faithfulness_scores_per_annotation_level_og.svg", "wb") as fid:
    fid.write(fig.to_image("svg"))

fig


In [ ]:
original = mobert_og["naturalness"].to_numpy()
shuffled = mobert_corrupted["naturalness"].to_numpy()
pairs = np.vstack([original, shuffled]).T
diff = original - shuffled

import scipy.stats

display(scipy.stats.pearsonr(original, diff))
go.Figure(go.Scatter(x=original, y=diff, mode="markers"))

In [ ]:
original = mobert_og["faithfulness"].to_numpy()
shuffled = mobert_corrupted["faithfulness"].to_numpy()
pairs = np.vstack([original, shuffled]).T
diff = original - shuffled

display(scipy.stats.pearsonr(original, diff))

go.Figure(go.Scatter(x=original, y=diff, mode="markers"))